> **💾 First: save your own copy** · This notebook opens from the course website, so Colab does not keep your changes. Before you do anything else, go to **File → Save a copy in Drive**. Colab opens your copy in a new tab: work in that tab and close this one. You find your copy later in Google Drive, in the folder *Colab Notebooks*.

![Cultural Data Analysis · Week 4: Cultural Data Forms: Text](https://goto4711.github.io/cda-teaching-tools/assets/notebooks/week4-banner.png)

# Are All LLMs the Same? Prompting through APIs

In this workshop, you learn how researchers talk to language models: not through the chat window, but through an **API**. And you use what you get back to study the models themselves as cultural objects.

We follow an experiment by AlgorithmWatch: [Are all LLMs the same?](https://algorithmwatch.org/en/are-all-llms-the-same-i-set-out-to-measure-their-similarity/) The author asked several models for "20 important dates in world history" and reports a narrow, US-centric, technical view of history that the models share. Can we reproduce this? And how much do our own measuring tools shape what we find?

| Part | What you do | Minutes |
|---|---|---|
| 1. Talking to a model through its API | build a request, read the answer, count tokens and costs | 15 |
| 2. The data and how we measure it | 150 answers from five models in three languages | 6 |
| 3. Are all LLMs the same? | compare the lists with the Jaccard index and tf-idf | 10 |
| 4. What is missing? | check an LLM's coding by hand, count regions, find places with spaCy | 15 |
| 5. Your own prompt | change one thing in the prompt | 2 |
| Reflection | three short answers | 2 |

> **⚠️ Important** · **This is a group exercise.** Work in groups of three or four. Your group chooses **one** of the two Week 4 notebooks: *Are All LLMs the Same?* (whose history do language models tell?) or *Frozen in Time* (can language models forecast world events?). Everyone runs the notebook on their own laptop, but you work through it together. Stop at the **💬 Discuss** boxes and talk before you go on. One API key per group is enough. Each member hands in their own notebook.

**Group members:** *write the names of your group here*

> **⚠️ Important** · You don't need an API key: without one, the notebook uses answers that we collected for you with exactly the same requests. With a free key you can send the requests yourself (see Part 1). Getting a key takes about five minutes, so do it before the workshop or at home afterwards.

Please run the cell below to set up the environment.

In [ ]:
#Keep cell
import os, re, json, random, urllib.request
from collections import Counter
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
for r in ['punkt', 'punkt_tab', 'stopwords']:
  nltk.download(r, quiet=True)
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer

try:
  import learntools.core
  from learntools.core import binder
  binder.bind(globals())
except:
  !pip install -q git+https://github.com/Kaggle/learntools.git
  import learntools.core
  from learntools.core import binder
  binder.bind(globals())

BASE = 'https://raw.githubusercontent.com/goto4711/cdai/main/'
for f in ['prompting_workshop_tests.py', 'llm_dates_2026.json']:
  if not os.path.exists(f):
    urllib.request.urlretrieve(BASE + f, f)

from prompting_workshop_tests import *

data = json.load(open('llm_dates_2026.json', encoding='utf-8'))
responses = pd.DataFrame(data['responses'])
events = pd.DataFrame(data['events'])
REGIONS, DOMAINS = data['about']['regions'], data['about']['domains']
print(f"{len(responses)} answers from {responses['label'].nunique()} models, collected on {data['about']['collected']}")

---

## 1. Talking to a model through its API (15 min)

An **API** (application programming interface) is how one program talks to another. For a language model, your program sends a **request** over the internet and gets a **response** back. The chat window of ChatGPT or Gemini does exactly this behind the scenes.

Researchers use the API rather than the chat window because they can

* send **exactly the same prompt** many times, and to several models;
* set and record the **settings**, such as the model version and the *temperature* (how much randomness);
* **save** every answer with its details, so others can check the work.

Note that the API is not the same as the chat app: the app adds its own instructions, memory and sometimes a web search. Through the API we get closer to the model itself (though the provider still filters and processes requests on its side).

A request has three parts:

1. a **URL**: the address of the model;
2. **headers**: who you are, i.e. your **API key**;
3. a **body**: what you send, i.e. the prompt and the settings, written as JSON (nested Python dictionaries and lists).

> **💡 Hint** · **Optional: your own free key.** Go to [aistudio.google.com](https://aistudio.google.com) → *Get API key*. In Colab, click the key symbol on the left → *Add new secret*, name it `GEMINI_API_KEY`, paste the key and switch on *Notebook access*. Read the terms of the free tier: Google may use what you send to improve its products, so never send personal data.

> **⚠️ Important** · Never paste an API key into your code. Anyone who sees the notebook can then use it, at your cost. Notebooks are shared all the time.

Run the next cell. It looks for a key in Colab's secrets.

In [ ]:
#Keep cell
try:
  from google.colab import userdata
  GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
  GEMINI_API_KEY = None

if GEMINI_API_KEY:
  print('Key found: this notebook will ask Gemini itself.')
else:
  print('No key: this notebook uses the pre-collected answers. Everything works the same.')

Now we build the request. Run the next cell to set the URL and the headers. Note how the key goes into the headers, not into the URL or the code.

In [ ]:
#Keep cell
PROMPT = data['about']['prompts']['en']
MODEL = data['about'].get('student_model', 'gemini-3.5-flash-lite')   # a small Gemini model with a free tier

url = f'https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent'
headers = {'Content-Type': 'application/json',
           'x-goog-api-key': GEMINI_API_KEY}

print('URL:', url)
print('\nPrompt:\n' + PROMPT)

The **body** is a nested dictionary. Gemini expects the prompt as `text` inside `parts` inside `contents`, and the settings in `generationConfig`.

Replace the two `____` below: put our `PROMPT` in the right place, and set the temperature to `1.0`. The **temperature** controls the randomness when the model picks the next word: low values make it pick the most likely words, `1.0` (the usual default) lets it vary. Google recommends keeping Gemini 3 models at `1.0`.

Complete the next cell.

In [ ]:
#Complete cell
body = {
    'contents': [{'parts': [{'text': ____}]}],
    'generationConfig': {'temperature': ____}
}

In [ ]:
q1.check()
#q1.hint()
#q1.solution()

Now we send the request with `requests.post()`. If you have no key, the cell takes a saved response to exactly this request from our data.

Run the next cell. The response is long, so we only print the start.

In [ ]:
#Keep cell
response = data['example_response']        # the saved answer, used if there is no key or the request fails
if GEMINI_API_KEY:
  r = requests.post(url, headers=headers, json=body, timeout=120)
  print('Status code:', r.status_code)
  if r.status_code == 200:
    response = r.json()
  else:
    print('Error message:', r.json().get('error', {}).get('message'), '\nWe use the saved answer instead.\n')

print(json.dumps(response, indent=2, ensure_ascii=False)[:1500])

The **status code** tells you whether it worked:

| Code | Meaning |
|---|---|
| `200` | OK |
| `400` | a mistake in the request (with Gemini also: a wrong key) |
| `403` | not allowed, e.g. a problem with the key |
| `404` | not found, e.g. a model name that does not exist (any more) |
| `429` | too many requests: wait a minute |
| `503` | the service is overloaded: try again later |

The response is JSON again: dictionaries inside lists inside dictionaries.

Find the model's answer in `response` and save it in `answer`. Follow the path in the printout: the key `'candidates'` holds a list; take its first item (`[0]`), then `'content'`, then `'parts'`, again the first item, then `'text'`.

Complete the next cell.

In [ ]:
#Complete cell
answer = ____
print(answer)

In [ ]:
q2.check()
#q2.hint()
#q2.solution()

Every request costs money, counted in **tokens**: pieces of words. In English, a token is roughly ¾ of a word; Dutch and especially Chinese need more tokens for the same content, so the same request costs more. The response says in `usageMetadata` how many tokens went in (your prompt) and came out. Output tokens cost more than input tokens. Models that "think" before answering also bill the thinking as output (`thoughtsTokenCount`).

Save the number of input tokens (key `'promptTokenCount'`) in `tokens_in` and the tokens of the answer (key `'candidatesTokenCount'`) in `tokens_answer`. The cell then adds the thinking tokens and computes the cost in dollars. The prices are per million tokens.

Complete the next cell.

In [ ]:
#Complete cell
PRICE_IN = 0.30    # dollars per 1 million input tokens (Gemini 3.5 Flash-Lite, October 2026)
PRICE_OUT = 2.50   # dollars per 1 million output tokens, thinking included

usage = response['usageMetadata']
tokens_in = ____
tokens_answer = ____
tokens_out = tokens_answer + usage.get('thoughtsTokenCount', 0)
cost = (tokens_in * PRICE_IN + tokens_out * PRICE_OUT) / 1_000_000

print(f'{tokens_in} tokens in, {tokens_out} tokens out: ${cost:.5f}')
print(f'Our data set has 150 such answers: about ${150 * cost:.2f}')

In [ ]:
q3.check()
#q3.hint()
#q3.solution()

**Tokens are not words.** In the lecture, you split texts into words with NLTK's `word_tokenize()`. Language models use their own tokenizer, which cuts rare words into smaller pieces (*sub-words*) and turns each piece into a number. Run the next cell to compare the two counts for the same answer.

In [ ]:
#Keep cell
words = word_tokenize(answer)
print(len(words), 'words and punctuation marks with word_tokenize(), as in the lecture')
print(tokens_answer, 'tokens billed by the API for the same answer')
print('First words:', words[:12])

Cheap for one experiment, but it adds up when you send thousands of texts.

Let's put the request into a function, `ask()`, so we can reuse it. Then we ask twice with the same prompt. Without a key, you get two of the pre-collected Gemini answers.

Run the next cell.

In [ ]:
#Keep cell
def ask(prompt, temperature=1.0):
  """Send one prompt to Gemini and return the text of the answer."""
  body = {'contents': [{'parts': [{'text': prompt}]}],
          'generationConfig': {'temperature': temperature}}
  r = requests.post(url, headers=headers, json=body, timeout=120)
  if r.status_code != 200:
    raise RuntimeError(f"{r.status_code}: {r.json().get('error', {}).get('message')}")
  parts = r.json()['candidates'][0]['content']['parts']
  return ''.join(p.get('text', '') for p in parts)   # the answer can come in several parts

gem = responses[(responses['label'] == 'Gemini') & (responses['language'] == 'en')]
first, second = gem['text'].iloc[0], gem['text'].iloc[1]     # two saved answers
if GEMINI_API_KEY:
  try:
    first, second = ask(PROMPT), ask(PROMPT)
  except Exception as e:
    print('The request did not work:', e, '- we use two saved answers instead.\n')

years_first = set(re.findall(r'\d{3,4}', first))
years_second = set(re.findall(r'\d{3,4}', second))
print('In both answers:     ', sorted(years_first & years_second))
print('Only in the first:   ', sorted(years_first - years_second))
print('Only in the second:  ', sorted(years_second - years_first))

Same model, same prompt, different lists. Why? And what does this mean for research that uses one answer from a chatbot?

> **💬 Discuss** · If someone in your group has a key, compare your lists: does each of you get a different answer from the same model? Together, decide what one answer from a chatbot is worth as evidence.

Work through the next cell to reveal the answer.

In [ ]:
q4.hint()
#q4.solution()

> **💡 Remember** · Other providers work the same way: a URL, a key in the headers, a JSON body with the prompt and settings, a JSON response. Only the names differ, e.g. where the answer is in the response:
>
> | Provider | Answer in the response | Tokens |
> |---|---|---|
> | Google Gemini | `['candidates'][0]['content']['parts'][0]['text']` | `['usageMetadata']` |
> | OpenAI (and Mistral, DeepSeek … which copy its format) | `['choices'][0]['message']['content']` | `['usage']` |
> | Anthropic Claude | `['content'][0]['text']` | `['usage']` |
>
> We collected our data through **OpenRouter**, a service that offers hundreds of models from different companies through one OpenAI-style API, with one key and one bill. That is convenient, but it puts another company between you and the model. Our data contains the full response of every request in the column `raw`: try `responses['raw'].iloc[0]`. Which format is it, and which company actually ran the model (`'provider'`)?

---

## 2. The data and how we measure it (6 min)

For this workshop we sent the same prompt to five models, ten times each, and in three languages: English, Dutch and Chinese. Run the next cell to see the data and the prompts.

In [ ]:
#Keep cell
print(pd.crosstab(responses['label'], responses['language']), '\n')
for m in data['about']['models']:
  print(f"{m['label']:9} = {m['model']}")
print()
for lang, p in data['about']['prompts'].items():
  print(f'--- {lang} ---\n{p}\n')

To compare the lists, we need the years. Last year's version of this notebook searched the whole answer for four-digit numbers from 1000 to 2029. Below, we compare it with a function, `parse_dates()`, that reads the answer line by line, including years before 1000 and dates before the common era (BCE, *v.Chr.*, 公元前). **Years BCE become negative numbers**: 221 BCE is `-221`.

You don't need to understand every line of `parse_dates()`. Run the next cell and compare the two results.

In [ ]:
#Keep cell
def old_years(text):
  """Last year's method: four-digit years from 1000 to 2029 anywhere in the text."""
  return sorted(set(int(y) for y in re.findall(r'\b(1[0-9]{3}|20[0-2][0-9])\b', text)))

import re

BCE = r'(BCE|B\.?\s?C\.?|v\.\s?Chr\.?|voor Christus|公元前|紀元前|前(?=\s*\d)|기원전)'


def parse_dates(text):
    """Turn an LLM answer into a list of (year, event) pairs.
    Years before the common era become negative numbers, e.g. 221 BCE -> -221."""
    dates = []
    for line in text.splitlines():
        line = line.replace('*', '').replace('｜', '|').replace('：', ':').strip()
        line = line.strip('|').strip()                       # rows of a markdown table
        if not re.search(r'\d', line) or re.fullmatch(r'[|:\-\s]+', line):
            continue
        if '|' in line:
            cells = [c.strip() for c in line.split('|')]
            # the year is in the first cell with a number (skip a column that only numbers the rows)
            i = next((k for k, c in enumerate(cells) if re.search(r'\d{3}|\d\s*(BCE|BC|v\.\s?Chr|年)', c)),
                     next((k for k, c in enumerate(cells) if re.search(r'\d', c)), 0))
            left, event = cells[i], ' | '.join(c for c in cells[i + 1:] if c)
        else:                                                # e.g. "1492 - Columbus" or "1492: Columbus"
            m = re.match(r'^(.{0,30}?\d+[^\s:–—-]*)\s*(?::\s*|[–—-]\s+)(.+)$', line)
            if not m:
                continue
            left, event = m.group(1), m.group(2)
        if re.search(r'century|eeuw|世纪|世紀', left, re.I):  # "3rd century BCE": no single year
            continue
        left = re.sub(r'^\d{1,2}[.)]\s+(?=.*\d)', '', left.strip())   # a list number like "3. 1492"
        left = left.replace(',', '')
        nums = re.findall(r'-?\d{1,5}', left)
        if not nums or not event:
            continue
        year = int(next((n for n in nums if len(n.lstrip('-')) >= 3), nums[0]))   # "4 July 1776" -> 1776
        if year > 0 and re.search(BCE, left, re.I):
            year = -year
        dates.append((year, event.strip()))
    return dates


example = responses[(responses['label'] == 'Gemini') & (responses['language'] == 'en')]['text'].iloc[0]
print(example, '\n')
print('Old method:', old_years(example))
print('New method:', [y for y, e in parse_dates(example)])

The data set already contains all events found with `parse_dates()`, in `events`. Let's measure how much the old method would have lost for each model. (It also made other mistakes: it read "1200 BCE" as 1200, and counted years mentioned inside the descriptions. We leave these aside.)

Mark the events that the old method can see in a new column `old_method_sees`: years between 1000 and 2029. Use `.between(1000, 2029)` on the `year` column. The next line then computes the share of each model's events that the old method misses.

Complete the next cell.

In [ ]:
#Complete cell
events['old_method_sees'] = ____
missed = 1 - events.groupby('label')['old_method_sees'].mean()
print(missed.sort_values().round(3))

In [ ]:
q5.check()
#q5.hint()
#q5.solution()

Which events disappear? Run the next cell.

In [ ]:
#Keep cell
lost = events[~events['old_method_sees']]
print(lost.groupby(['year', 'event']).size().sort_values(ascending=False).head(15))

Look at the list: whose history are these lost events? Ancient Rome and Greece, or also China, India, Egypt and the beginnings of Islam? Which model loses the most, and what does that do to a comparison of the models?

> **💡 Remember** · A measuring tool can have its own bias. Last year's regular expression quietly dropped every event before the year 1000. Always check what your method throws away.

---

## 3. Are all LLMs the same? (10 min)

The article compares two lists with the **Jaccard index**: the number of years that are in both lists, divided by the number of years in either list. 1 means the same years, 0 means no year in common.

Run the next cell. It makes one set of years for every answer, and defines `jaccard()`.

In [ ]:
#Keep cell
runs = events.groupby(['label', 'language', 'run'])['year'].apply(set).reset_index()

def jaccard(a, b):
  if not a | b:            # two empty lists
    return 0.0
  return len(a & b) / len(a | b)

print('Answers per model and language (an answer without any readable date is left out):')
print(pd.crosstab(runs['label'], runs['language']))
runs.head()

Not every answer is usable: some models ignored the format or got stuck repeating one character. If a count in the table is below 10, read that answer later, e.g. `responses[(responses['label'] == 'Gemini') & (responses['language'] == 'zh')]['text'].iloc[8][:300]`. A real study has to report such failures.

Are two answers from the **same model** more alike than two answers from **different models**? The loop below compares every pair of English answers. Replace `____` with a condition that is `True` when both answers come from the same model (column `label`).

Complete the next cell.

In [ ]:
#Complete cell
en = runs[runs['language'] == 'en'].to_dict('records')
within, between = [], []
for i in range(len(en)):
  for j in range(i + 1, len(en)):
    score = jaccard(en[i]['year'], en[j]['year'])
    if ____:
      within.append(score)
    else:
      between.append(score)

print(f'Same model:       {np.mean(within):.2f}')
print(f'Different models: {np.mean(between):.2f}')

In [ ]:
q6.check()
#q6.hint()
#q6.solution()

A heatmap shows this for every pair of models: the average Jaccard index of their English answers. The diagonal compares a model with itself (different runs).

Run the next cell.

In [ ]:
#Keep cell
labels = sorted(runs['label'].unique())
sim = pd.DataFrame(index=labels, columns=labels, dtype=float)
for a in labels:
  for b in labels:
    sim.loc[a, b] = np.mean([jaccard(x['year'], y['year']) for x in en for y in en
                             if x['label'] == a and y['label'] == b and x is not y])

sns.heatmap(sim, annot=True, fmt='.2f', cmap='YlOrRd', vmin=0, vmax=1, square=True)
plt.title('Average Jaccard index of the English answers')
plt.show()

> **💬 Discuss** · Does this support the article's claim that *all LLMs are the same*? Agree on an answer as a group and keep it for the reflection.

 Which pairs of models are closest? And which years form the shared core, appearing in at least three quarters of all English answers? Run the next cell.

In [ ]:
#Keep cell
counts = Counter(y for r in en for y in r['year'])
core = sorted(y for y, n in counts.items() if n >= 0.75 * len(en))
print(f'Years in at least 75% of the {len(en)} English answers:')
print(events[events['year'].isin(core) & (events['language'] == 'en')].groupby('year')['event'].first())

The Jaccard index compares years. What about the words? Which words are typical for each model's history? This is a job for **tf-idf** from the lecture: it finds words that are frequent in one document but not in all of them. We treat all English event descriptions of one model as one document, and use the same settings as in the lecture.

Run the next cell.

In [ ]:
#Keep cell
model_docs = events[events['language'] == 'en'].groupby('label')['event'].apply(' '.join)

tfidf = TfidfVectorizer(lowercase = True, stop_words = 'english', token_pattern = '[a-zA-Z][a-zA-Z][a-zA-Z]+')
vec = tfidf.fit_transform(model_docs)
tfidf_df = pd.DataFrame(vec.toarray().transpose(), index=tfidf.get_feature_names_out(), columns=model_docs.index)

for m in tfidf_df.columns:
  print(f"{m:9}", list(tfidf_df[m].sort_values(ascending=False).head(8).index))

Does each model have its own vocabulary of history: revolutions, wars, inventions, empires? Words that all models use (e.g. *war*) get a low score.

---

## 4. What is missing? (15 min)

To say whose history is missing, we need to know where each event happened and what it is about. Coding 3,000 events by hand takes days. So we did what many researchers now do: we let an LLM code them. Studies report that LLMs can annotate texts as well as human coders (Törnberg 2023), but also that this has to be checked for every new task (Pangakis et al. 2023). Each event in `events` has a `region` and a `domain`.

Can we trust this coding? Note that the coding model also codes its own answers. Here is the prompt that was used. Run the next cell.

In [ ]:
#Keep cell
print('Coded by', data['about']['coder']['model'], '\n')
print(data['about']['coding_prompt'])

Let's check a sample by hand, as you would in a real research project. Run the next cell to see ten events.

In [ ]:
#Keep cell
sample = events[events['language'] == 'en'].drop_duplicates('event').sample(10, random_state=4711)
for i, (y, e) in enumerate(zip(sample['year'], sample['event'])):
  print(f'{i}: {y} | {e}')
print('\nRegions:', REGIONS)

Code the ten events yourself, **alone**, before you look at your group's or the LLM's codes. Write one region per event in the list below, in the same order, spelled exactly as in `REGIONS`.

Complete the next cell.

In [ ]:
#Complete cell
my_regions = ['___', '___', '___', '___', '___', '___', '___', '___', '___', '___']

In [ ]:
q7.check()
#q7.hint()
#q7.solution()

Now compare your codes with the LLM's. Run the next cell.

In [ ]:
#Keep cell
compare = pd.DataFrame({'year': sample['year'].values, 'event': sample['event'].values,
                        'you': my_regions, 'LLM': sample['region'].values})
compare['agree'] = compare['you'] == compare['LLM']
print(f"You and the LLM agree on {compare['agree'].mean():.0%} of the events.\n")
compare[~compare['agree']]

> **💬 Discuss** · First compare your own codes **within the group**: on how many of the ten events do you agree with each other? Then compare with the LLM. If the humans disagree among themselves, what does it mean to say the LLM is "wrong"? Where did the Second World War "mainly" take place?

Who is right, you or the LLM, or is the category itself unclear? In a real project, you would code a larger sample (e.g. 100 events), report the agreement, and change the categories or the prompt if it is too low.

Now let's count. Which regions does each model's history cover? Make a table with one row per model and one column per region, showing the **share** of the English events in each region. `pd.crosstab()` counts how often each combination of two columns occurs; with `normalize='index'` it gives shares per row.

Complete the next cell.

In [ ]:
#Complete cell
en_events = events[events['language'] == 'en']
shares = ____
shares.round(2)

In [ ]:
q8.check()
#q8.hint()
#q8.solution()

Run the next cell to plot it.

In [ ]:
#Keep cell
shares.plot(kind='barh', stacked=True, figsize=(10, 4), colormap='tab10')
plt.legend(bbox_to_anchor=(1, 1))
plt.xlabel('Share of the events')
plt.title('Where do the English "important dates in world history" take place?')
plt.show()

Does the language of the prompt change the picture? Make the same table, but now with one row per **language** (for all models together), and save it in `lang_shares`.

Complete the next cell.

In [ ]:
#Complete cell
lang_shares = ____
lang_shares.round(2)

In [ ]:
q9.check()
#q9.hint()
#q9.solution()

### A second tool: places with spaCy

In the lecture, spaCy's named entity recognition found the places (`GPE`, *geopolitical entities*) in Bush's State of the Union. Let's use it as a second, rule-based way to see which places the models' histories mention, and compare it with the LLM's coding. Run the next cell; it runs spaCy on all English event descriptions.

In [ ]:
#Keep cell
import spacy
nlp = spacy.load('en_core_web_sm')
docs = list(nlp.pipe(events[events['language'] == 'en']['event']))
print([(ent.text, ent.label_) for ent in docs[0].ents])

Collect the text of every `GPE` entity in a list `places`. The loop goes through all descriptions; inside, a list comprehension picks the entities, as in the lecture: `[ent for ent in doc.ents if ent.label_ in ['GPE']]`. Replace `____` so that it keeps the text of each entity (`ent.text`).

Complete the next cell.

In [ ]:
#Complete cell
places = []
for doc in docs:
  places += [____ for ent in doc.ents if ent.label_ == 'GPE']

Counter(places).most_common(15)

In [ ]:
q11.check()
#q11.hint()
#q11.solution()

Compare this with the LLM's region coding. What does spaCy find that the regions hide (which countries exactly)? Where does it go wrong? Look for people or events labelled as places, and for places it misses. Both are tools with their own errors: spaCy follows patterns it learned from news texts, the LLM follows a prompt.

> **💡 Remember** · We have now used three measuring tools whose results we must check: a parser (Part 2), an LLM as coder and spaCy. With the hand-coding you have done the most important step of using an LLM in research: **validation**. You will need this in the group project if you use an LLM as a method.

---

## 5. Your own prompt (2 min)

As a group, agree on **one** thing to change in the prompt, and see what happens. For example:

* the topic: *20 important dates in the history of women's rights / of the internet / of Southeast Asia*;
* the perspective: *… as a history teacher in Lagos / in Jakarta would choose them*;
* the number: *5* instead of *20* dates.

Keep the last two lines of the prompt (the format, in English), so that `parse_dates()` can still read the answer. Write your prompt in `my_prompt`.

Complete the next cell.

In [ ]:
#Complete cell
my_prompt = '''___
Format strictly as: YEAR | EVENT DESCRIPTION
One entry per line. No extra text.'''

In [ ]:
q10.check()
#q10.hint()
#q10.solution()

With a key, the next cell asks Gemini three times and compares the years with the answers to the original prompt. Without a key, write down what you expect, and try it later with a free key.

Run the next cell.

In [ ]:
#Keep cell
if GEMINI_API_KEY:
  try:
    mine = [set(y for y, e in parse_dates(ask(my_prompt))) for _ in range(3)]
    original = [set(y for y, e in parse_dates(ask(PROMPT))) for _ in range(3)]
    print('Your prompt, 3 answers - years in all three:', sorted(set.intersection(*mine)))
    print(f'Similarity among your answers:            {np.mean([jaccard(a, b) for a in mine for b in mine if a is not b]):.2f}')
    print(f'Similarity with the original prompt:      {np.mean([jaccard(a, b) for a in mine for b in original]):.2f}')
  except Exception as e:
    print('The request did not work:', e)
else:
  print('No key: what do you expect? Note it in the reflection below.')

---

## Reflection (2 min)

Discuss the three questions in your group, then each write your answers in two or three sentences.

1. **Are all LLMs the same?** Compare the Jaccard index within and between models. Does your result support the article's claim?
2. **Whose history?** Which regions are missing or rare, and does the language of the prompt change this?
3. **Our instruments:** Where did our own tools (the parser, the LLM coding, the categories) shape what we found?

*Your answers*

### If you have time: extensions

* **Embeddings:** the Jaccard index only matches identical years. "Gutenberg's printing press" may be dated 1440 or 1450. How could the sentence embeddings from the lecture help to match events that are the same but described or dated differently?
* **Another provider:** get a key from another provider and change `url`, `headers` and the path to the answer, using the table in Part 1.

> **⚠️ Important** · Run the next cell to create an automated HTML download of your notebook for submission to Canvas. Please **make sure you have saved the notebook** before running the code, and check the HTML file before you submit it.

In [ ]:
#Keep cell
import os

if not(os.path.exists('make_html.py')):
  print('Downloading make_html.py\n')
  !wget -q https://raw.githubusercontent.com/goto4711/cdai/refs/heads/main/make_html.py

from make_html import make_html

make_html("Week4_Prompting.ipynb")